# Fine-tune original models on synthetic data

This notebook demonstrates how to fine-tune the original models (SmolVLM, Granite, Gemma, Ministral) using the synthetically generated training data.

The workflow:
1. Submit fine-tuning jobs to Azure ML for each model
2. Wait for jobs to complete
3. Auto-discover the resulting checkpoints from the model registry
4. List the new fine-tuned models for use in downstream validation tasks

In [1]:
import subprocess
import json
from pathlib import Path
from datetime import datetime, timezone
from posixpath import normpath

# Define the models to fine-tune.
# Each entry contains: (name, model_slug, epochs, batch_size)
MODEL_JOBS = [
    ("SmolVLM", "smolvlm2", 5, 1),
    ("Granite4", "granite4", 5, 1),
    ("Gemma3", "gemma3", 5, 1),
    ("Gemma4", "gemma4", 5, 1),
    ("Ministral", "ministral", 5, 1),
]

# Multi-GPU fine-tune settings for Standard_ND96amsr_A100_v4
FINETUNE_GPU_WORKERS = 8
GRAD_ACCUM_STEPS = 8
AUTO_SCALE_GRAD_ACCUM = True

# Dataset configuration
# Option A: Set a quick selector (real/fake/test_real/test_fake)
DATASET_NAME = "fake"
# Option B: Set a custom dataset root containing images/ and transcriptions/
# Example: "fake_daily_rainfall_v2"
DATASET_DIR = "fake_daily_rainfall_2"

if DATASET_DIR:
    TRAINING_IMAGES_PATH = f"{DATASET_DIR.rstrip('/')}/images"
else:
    DATASET_IMAGES_MAP = {
        "real": "Daily_rainfall_sample/images",
        "fake": "fake_daily_rainfall/images",
        "test_real": "test_data/real/images",
        "test_fake": "test_data/fake/images",
    }
    if DATASET_NAME not in DATASET_IMAGES_MAP:
        raise ValueError(f"Unknown DATASET_NAME: {DATASET_NAME}")
    TRAINING_IMAGES_PATH = DATASET_IMAGES_MAP[DATASET_NAME]

print(f"Configured to fine-tune {len(MODEL_JOBS)} models")
print(f"Training images path: {TRAINING_IMAGES_PATH}")
print(f"Finetune GPU workers: {FINETUNE_GPU_WORKERS}")
print(f"Base grad accumulation steps: {GRAD_ACCUM_STEPS}")
print(f"Auto-scale grad accumulation: {AUTO_SCALE_GRAD_ACCUM}")

Configured to fine-tune 5 models
Training images path: fake_daily_rainfall_2/images
Finetune GPU workers: 8
Base grad accumulation steps: 8
Auto-scale grad accumulation: True


## 1. Submit fine-tuning jobs

The cell below submits one fine-tuning job to Azure ML for each of the five models.
Each job will:
- Load the original pre-trained model
- Train on the selected dataset (`DATASET_NAME`) or custom dataset directory (`DATASET_DIR`)
- Save the LoRA adapter checkpoint

Jobs run in parallel on GPU nodes. Depending on queue and resource availability, this may take 30 mins - 2 hours total.

In [2]:
submitted_runs = []

for model_name, model_slug, epochs, batch_size in MODEL_JOBS:
    print(f"\nSubmitting fine-tuning job for {model_name}...")

    cmd = [
        "bash",
        "../scripts/aml_submit.sh",
    ]

    if DATASET_DIR:
        cmd.extend(["--dataset-dir", DATASET_DIR])
    else:
        cmd.extend(["--dataset", DATASET_NAME])

    cmd.extend(
        [
            "--model",
            model_slug,
            "--epochs",
            str(epochs),
            "--batch-size",
            str(batch_size),
            "--finetune-gpu-workers",
            str(FINETUNE_GPU_WORKERS),
            "--grad-accum-steps",
            str(GRAD_ACCUM_STEPS),
            "--auto-scale-grad-accum",
            "true" if AUTO_SCALE_GRAD_ACCUM else "false",
            "finetune",
        ]
    )

    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    print(result.stdout)
    if result.returncode != 0:
        print(f"Warning: Job submission failed for {model_name}")
        print(result.stderr)
    else:
        submitted_runs.append(model_slug)

print(f"\n✓ Submitted {len(submitted_runs)} fine-tuning jobs")
print("Jobs will run in the background. Check Azure ML UI for progress.")


Submitting fine-tuning job for SmolVLM...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm2
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/fake_daily_rainfall_2/images
Transcript: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/fake_daily_rainfall_2/transcriptions
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting finetune job...
  Checkpoint output:

## 2. Wait for jobs to complete

Once the fine-tuning jobs have completed, the resulting checkpoints will be automatically registered in the model registry.

You can monitor job progress in the Azure ML workspace UI, or run the cell below to check the status.

In [23]:
# Optional: Monitor job progress
# Uncomment and run to see job status

# for model_slug in submitted_runs:
#     print(f"Checking status for {model_slug}...")
#     subprocess.run(
#         ["az", "ml", "job", "show", "-n", f"finetune-{model_slug}"],
#         check=True
#     )

## 2a. Track training metrics across models

Run this cell to query Azure ML and compare the status, creation time, and duration of fine-tuning jobs across all five models.
This helps identify which models are training successfully and which ones may be struggling.

## 2b. Extract training metrics from job logs

This cell retrieves the actual training logs from Azure ML and extracts key performance metrics:
- **Final Loss**: The training loss value (lower is better; should decrease over epochs)
- **Eval Loss**: Validation loss if available 
- **Token Accuracy**: Percentage of correctly predicted tokens in the output

This allows you to compare which models are learning effectively. If granite4's loss isn't decreasing or token accuracy is very low compared to other models, it indicates a training issue.

In [3]:
import subprocess
import pandas as pd
from datetime import datetime

# Track training metrics across all models
print("=" * 80)
print("TRAINING METRICS ACROSS MODELS")
print("=" * 80)

metrics_data = []

for model_name, model_slug, _, _ in MODEL_JOBS:
    # Query Azure ML for the most recent job for this model on the selected dataset
    cmd = [
        "az", "ml", "job", "list",
        "--workspace-name", "mlw-llmdatarescue-uksouth-01",
        "--resource-group", "rg-climate-llmdatarescue",
        "--subscription", "79c7890c-2a30-44ef-aa8d-419d25b7bb8e",
        "--query", f"[?contains(display_name,'finetune-{model_slug}')] | sort_by(@,&creation_context.created_at)[-1]",
        "-o", "json"
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and result.stdout.strip() and result.stdout.strip() != "null":
        try:
            job = json.loads(result.stdout)
            if job:
                status = job.get("status", "Unknown")
                created = job.get("creation_context", {}).get("created_at", "N/A")
                duration = job.get("duration", None)
                
                # Extract job ID for potential log inspection
                job_id = job.get("name", "N/A")
                
                # Parse duration if available (format: "HH:MM:SS")
                duration_str = duration if duration else "N/A"
                
                metrics_data.append({
                    "Model": model_name,
                    "Status": status,
                    "Job ID": job_id[:16],  # truncate for readability
                    "Created": created[:10] if created != "N/A" else "N/A",
                    "Duration": duration_str
                })
        except json.JSONDecodeError:
            metrics_data.append({
                "Model": model_name,
                "Status": "Error parsing response",
                "Job ID": "N/A",
                "Created": "N/A",
                "Duration": "N/A"
            })
    else:
        metrics_data.append({
            "Model": model_name,
            "Status": "No job found",
            "Job ID": "N/A",
            "Created": "N/A",
            "Duration": "N/A"
        })

if metrics_data:
    df_metrics = pd.DataFrame(metrics_data)
    print("\n" + df_metrics.to_string(index=False))
    print("\n" + "=" * 80)
    
    # Summary
    statuses = df_metrics["Status"].value_counts()
    print("\nStatus Summary:")
    for status, count in statuses.items():
        print(f"  {status}: {count}")
else:
    print("\nNo metrics data available. Submit jobs first.")

TRAINING METRICS ACROSS MODELS

    Model    Status           Job ID    Created Duration
  SmolVLM Completed goofy_garage_zx8 2026-06-23      N/A
 Granite4 Completed jolly_dinner_9mj 2026-06-23      N/A
   Gemma3 Completed joyful_melon_h73 2026-06-23      N/A
   Gemma4 Completed olive_soursop_t3 2026-06-23      N/A
Ministral   Running plucky_bee_85htz 2026-06-23      N/A


Status Summary:
  Completed: 4
  Running: 1


In [4]:
import re
import ast
import tempfile
import os
import glob

# Extract training metrics (loss, token accuracy) from job artifacts/log stream
print("=" * 100)
print("TRAINING PERFORMANCE METRICS (Loss & Token Accuracy)")
print("=" * 100)

training_metrics = []
epoch_history_rows = []


def _to_float(val):
    """Convert string or numeric value to float, or return None."""
    try:
        return float(val)
    except (TypeError, ValueError):
        return None


def _parse_metric_dicts(text):
    """Parse HuggingFace Trainer metric dicts from log text.

    Uses re.DOTALL so dicts that are wrapped across multiple lines
    (e.g. appended to a tqdm progress bar line) are still matched.
    Values may be strings ('0.6836') or floats (0.6836).
    Returns (train_entries, eval_entries).
    """
    train_entries = []
    eval_entries = []
    # Match any {...} block including newlines inside it.
    for m in re.finditer(r"\{[^{}]+\}", text, re.DOTALL):
        raw = m.group()
        if "loss" not in raw:
            continue
        try:
            entry = ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            continue
        if not isinstance(entry, dict):
            continue
        if "eval_loss" in entry:
            eval_entries.append(entry)
        elif "loss" in entry:
            train_entries.append(entry)
    return train_entries, eval_entries


def _collect_logs_from_download(job_id):
    """Return (log_text, source_note) using az ml job download.
    
    Handles preempted/restarted runs by collecting logs from:
    - Main artifacts directory (initial run)
    - retry_001, retry_002, etc. subdirectories (preemption restarts)
    """
    with tempfile.TemporaryDirectory() as tmpdir:
        cmd_download = [
            "az", "ml", "job", "download", "-n", job_id,
            "--workspace-name", "mlw-llmdatarescue-uksouth-01",
            "--resource-group", "rg-climate-llmdatarescue",
            "--subscription", "79c7890c-2a30-44ef-aa8d-419d25b7bb8e",
            "--download-path", tmpdir,
        ]
        dl = subprocess.run(cmd_download, capture_output=True, text=True, timeout=180)

        if dl.returncode != 0:
            return "", "download failed"

        # Collect logs from main artifacts and all retry directories
        all_logs = []
        sources = []
        
        # Look for std_log.txt in the main directory first
        std_log_path = None
        for root, dirs, files in os.walk(tmpdir):
            for fname in files:
                if fname == "std_log.txt":
                    std_log_path = os.path.join(root, fname)
                    break
            if std_log_path:
                break

        if std_log_path and os.path.exists(std_log_path):
            try:
                with open(std_log_path, encoding="utf-8", errors="replace") as f:
                    log_text = f.read()
                    if log_text:
                        all_logs.append(log_text)
                        sources.append("main")
            except OSError:
                pass
        
        # Look for retry_XXX directories (preemption/restart)
        retry_dirs = sorted(glob.glob(os.path.join(tmpdir, "**/retry_*"), recursive=True))
        for retry_dir in retry_dirs:
            if os.path.isdir(retry_dir):
                retry_name = os.path.basename(retry_dir)
                
                # Look for std_log.txt in the retry directory
                retry_std_log = os.path.join(retry_dir, "std_log.txt")
                if os.path.exists(retry_std_log):
                    try:
                        with open(retry_std_log, encoding="utf-8", errors="replace") as f:
                            log_text = f.read()
                            if log_text:
                                all_logs.append(log_text)
                                sources.append(retry_name)
                    except OSError:
                        pass
                
                # Also collect any other txt files in retry directory
                for fname in sorted(os.listdir(retry_dir)):
                    if fname.endswith(".txt") and fname != "std_log.txt":
                        fpath = os.path.join(retry_dir, fname)
                        try:
                            with open(fpath, encoding="utf-8", errors="replace") as f:
                                log_text = f.read()
                                if log_text:
                                    all_logs.append(log_text)
                                    sources.append(f"{retry_name}/{fname}")
                        except OSError:
                            pass
        
        if not all_logs:
            # Fallback: concatenate all txt files from entire download
            chunks = []
            txt_count = 0
            for root, dirs, files in os.walk(tmpdir):
                for fname in sorted(files):
                    if fname.endswith(".txt"):
                        txt_count += 1
                        fpath = os.path.join(root, fname)
                        try:
                            with open(fpath, encoding="utf-8", errors="replace") as f:
                                chunks.append(f.read())
                        except OSError:
                            pass
            if chunks:
                merged_log = "\n".join(chunks)
                source_note = f"fallback scan ({txt_count} txt files)"
                return merged_log, source_note
            else:
                return "", "no logs found"
        
        merged_log = "\n".join(all_logs)
        source_note = " + ".join(sources) if sources else "no logs collected"
        return merged_log, source_note


def _collect_logs_from_stream(job_id):
    """Return (log_text, source_note) using az ml job stream for running jobs."""
    cmd_stream = [
        "timeout", "35",
        "az", "ml", "job", "stream", "-n", job_id,
        "--workspace-name", "mlw-llmdatarescue-uksouth-01",
        "--resource-group", "rg-climate-llmdatarescue",
        "--subscription", "79c7890c-2a30-44ef-aa8d-419d25b7bb8e",
    ]
    try:
        st = subprocess.run(cmd_stream, capture_output=True, text=True, timeout=60)
        text = (st.stdout or "") + "\n" + (st.stderr or "")
        return text, "job stream (live)"
    except subprocess.TimeoutExpired:
        return "", "stream timeout"


def _epoch_value(entry):
    return _to_float(entry.get("epoch"))


def _fmt(x):
    return f"{x:.4f}" if isinstance(x, float) else "N/A"


for model_name, model_slug, _, _ in MODEL_JOBS:
    cmd_list = [
        "az", "ml", "job", "list",
        "--workspace-name", "mlw-llmdatarescue-uksouth-01",
        "--resource-group", "rg-climate-llmdatarescue",
        "--subscription", "79c7890c-2a30-44ef-aa8d-419d25b7bb8e",
        "--query",
        f"[?contains(display_name,'finetune-{model_slug}')] | sort_by(@,&creation_context.created_at)[-1]",
        "-o", "json",
    ]

    result_list = subprocess.run(cmd_list, capture_output=True, text=True)
    job_meta = None
    job_id = None
    if result_list.returncode == 0 and result_list.stdout.strip() not in ("", "null"):
        try:
            job_meta = json.loads(result_list.stdout)
            job_id = job_meta.get("name")
        except json.JSONDecodeError:
            pass

    if not job_id:
        training_metrics.append({
            "Model": model_name, "Status": "No job found",
            "Steps": "N/A", "Final Loss": "N/A", "Eval Loss": "N/A",
            "Token Accuracy": "N/A", "Epoch": "N/A", "Source": "N/A",
        })
        continue

    job_status = (job_meta or {}).get("status", "Unknown")

    # Collect logs from job artifacts, including retry directories from preemption
    if job_status in ("Completed", "Failed", "Canceled", "NotResponding", "Paused"):
        log_text, source_note = _collect_logs_from_download(job_id)
    else:
        log_text, source_note = _collect_logs_from_stream(job_id)

    train_entries, eval_entries = _parse_metric_dicts(log_text)

    # Sort by epoch for stable per-epoch matching.
    train_entries = sorted(train_entries, key=lambda e: (_epoch_value(e) or -1.0))
    eval_entries  = sorted(eval_entries,  key=lambda e: (_epoch_value(e) or -1.0))

    if train_entries:
        last_train = train_entries[-1]
        final_loss  = _to_float(last_train.get("loss"))
        final_loss_str = f"{final_loss:.4f}" if final_loss is not None else "N/A"
        final_epoch = last_train.get("epoch", "?")
        steps_seen  = len(train_entries)
        train_acc   = _to_float(last_train.get("mean_token_accuracy"))
    else:
        final_loss_str = "N/A"
        final_epoch = "N/A"
        steps_seen  = "N/A"
        train_acc   = None

    if eval_entries:
        last_eval = eval_entries[-1]
        eval_loss  = _to_float(last_eval.get("eval_loss"))
        eval_loss_str = f"{eval_loss:.4f}" if eval_loss is not None else "N/A"
        eval_acc   = _to_float(last_eval.get("eval_mean_token_accuracy"))
        token_accuracy_str = f"{eval_acc:.4f}" if eval_acc is not None else "N/A"
        final_epoch = last_eval.get("epoch", final_epoch)
    else:
        eval_loss_str = "N/A"
        token_accuracy_str = f"{train_acc:.4f}" if train_acc is not None else "N/A"

    training_metrics.append({
        "Model": model_name, "Status": job_status, "Steps": steps_seen,
        "Final Loss": final_loss_str, "Eval Loss": eval_loss_str,
        "Token Accuracy": token_accuracy_str, "Epoch": str(final_epoch),
        "Source": source_note,
    })

    # Per-epoch history: one row per eval pass, with nearest preceding train snapshot.
    # Prepend an epoch-0 row using the very first logged training step as a pre-training proxy.
    if train_entries:
        ft = train_entries[0]
        epoch_history_rows.append({
            "Model": model_name,
            "Status": job_status,
            "Epoch": 0,
            "Train Loss": _fmt(_to_float(ft.get("loss"))),
            "Train Acc":  _fmt(_to_float(ft.get("mean_token_accuracy"))),
            "Eval Loss":  "N/A",
            "Eval Acc":   "N/A",
            "Source": f"{source_note} (first step ep~{ft.get('epoch','?')})",
        })

    if eval_entries:
        for e in eval_entries:
            ep = _epoch_value(e)
            if ep is None:
                continue
            # Find the last train entry at or just before this eval epoch.
            train_at_epoch = None
            for t in train_entries:
                te = _epoch_value(t)
                if te is not None and te <= ep:
                    train_at_epoch = t
                elif te is not None and te > ep:
                    break

            tr_loss = _to_float(train_at_epoch.get("loss")) if train_at_epoch else None
            tr_acc  = _to_float(train_at_epoch.get("mean_token_accuracy")) if train_at_epoch else None
            ev_loss = _to_float(e.get("eval_loss"))
            ev_acc  = _to_float(e.get("eval_mean_token_accuracy"))

            epoch_history_rows.append({
                "Model": model_name,
                "Status": job_status,
                "Epoch": int(round(ep)),
                "Train Loss": _fmt(tr_loss),
                "Train Acc":  _fmt(tr_acc),
                "Eval Loss":  _fmt(ev_loss),
                "Eval Acc":   _fmt(ev_acc),
                "Source": source_note,
            })
    elif train_entries:
        # No eval yet (job still in first epoch): show latest train snapshot.
        lt = train_entries[-1]
        epoch_history_rows.append({
            "Model": model_name, "Status": job_status,
            "Epoch": lt.get("epoch", "N/A"),
            "Train Loss": _fmt(_to_float(lt.get("loss"))),
            "Train Acc":  _fmt(_to_float(lt.get("mean_token_accuracy"))),
            "Eval Loss": "N/A", "Eval Acc": "N/A",
            "Source": source_note,
        })

if training_metrics:
    df_training = pd.DataFrame(training_metrics)
    print("\nLATEST METRICS PER MODEL")
    print(df_training.to_string(index=False))

    if epoch_history_rows:
        df_epochs = pd.DataFrame(epoch_history_rows)
        model_order = {name: i for i, (name, _, _, _) in enumerate(MODEL_JOBS)}
        df_epochs["_mo"] = df_epochs["Model"].map(model_order)
        df_epochs["_ep"] = pd.to_numeric(df_epochs["Epoch"], errors="coerce")
        df_epochs = df_epochs.sort_values(["_mo", "_ep"]).drop(columns=["_mo", "_ep"])

        print("\n" + "=" * 100)
        print("EPOCH-BY-EPOCH METRICS")
        print(df_epochs.to_string(index=False))

    print("\n" + "=" * 100)
    print("\nInterpretation:")
    print("- Completed jobs: metrics from std_log.txt (main + retry_XXX directories)")
    print("- Preempted runs: logs automatically merged from all retry checkpoints")
    print("- Running jobs: metrics from live job stream (fewer epochs until next eval pass)")
    print("- Eval Loss/Acc should improve (decrease/increase) across epochs")
else:
    print("\nNo training metrics available yet.")

TRAINING PERFORMANCE METRICS (Loss & Token Accuracy)

LATEST METRICS PER MODEL
    Model    Status  Steps Final Loss Eval Loss Token Accuracy Epoch Source
  SmolVLM Completed     56     0.0027    0.0045         0.9986     5   main
 Granite4 Completed     56     3.9130    3.8770         0.6634     5   main
   Gemma3 Completed     56     0.6739    0.6733         0.9994     5   main
   Gemma4 Completed     56     0.0010    0.0017         0.9995     5   main
Ministral Completed     56     0.1255    0.1250         0.9820     5   main

EPOCH-BY-EPOCH METRICS
    Model    Status  Epoch Train Loss Train Acc Eval Loss Eval Acc                      Source
  SmolVLM Completed      0     0.5530    0.8384       N/A      N/A main (first step ep~0.0885)
  SmolVLM Completed      1     0.0080    0.9974    0.0079   0.9973                        main
  SmolVLM Completed      2     0.0085    0.9972    0.0058   0.9981                        main
  SmolVLM Completed      3     0.0043    0.9986    0.0049   0

## 3. Auto-discover fine-tuned checkpoints

When the jobs have completed, their checkpoints are registered in the model registry.
Run the cell below to discover the checkpoint paths for the newly fine-tuned models.
These checkpoints can then be used in downstream validation or extraction tasks.

In [ ]:
# Auto-discover checkpoints from the model registry

def _first_existing_path(candidates: list[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

REGISTRY_PATH = _first_existing_path(
    [Path("outputs/model_registry.json"), Path("../outputs/model_registry.json")]
)

def _norm_rel(path_like: str) -> str:
    return normpath(path_like.replace("\\", "/")).lstrip("./")

def _parse_created_at(value: str) -> datetime:
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)

registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
entries = registry.get("checkpoints", [])

CHECKPOINTS = {}
missing = []
expected_dataset_norm = _norm_rel(TRAINING_IMAGES_PATH)

for model_name, model_slug, _, _ in MODEL_JOBS:
    # Find all checkpoints for this model trained on the selected dataset path
    candidates = [
        e
        for e in entries
        if str(e.get("model_slug", "")) == model_slug
        and _norm_rel(str(e.get("training_dataset_path", ""))) == expected_dataset_norm
        and e.get("checkpoint_path")
    ]

    if not candidates:
        missing.append(model_name)
        continue

    # Get the most recently created checkpoint
    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )

    checkpoint_path = str(best["checkpoint_path"])
    CHECKPOINTS[model_slug] = checkpoint_path
    print(f"{model_name}: {checkpoint_path}")

if missing:
    print(f"\nWarning: Missing checkpoints for: {', '.join(missing)}")
    print("Wait for fine-tuning jobs to complete, then re-run this cell.")
else:
    print(f"\nFound all {len(CHECKPOINTS)} fine-tuned checkpoints")

## 4. Use fine-tuned models in validation

The fine-tuned checkpoints are now available for use in validation or extraction tasks.

To validate the fine-tuned models against the Ciara test data, see `validate_1st_order_consensus.ipynb`.

To extract full dataset with the fine-tuned models, use the extraction scripts with:
```bash
bash scripts/aml_submit.sh --checkpoint <checkpoint_path> --images-path ... extract
```

In [ ]:
# Display summary of fine-tuned checkpoints
if CHECKPOINTS:
    print("Fine-tuned checkpoints ready for use:\n")
    for model_slug, checkpoint_path in CHECKPOINTS.items():
        print(f"  {model_slug}:")
        print(f"    Path: {checkpoint_path}")
        print()
else:
    print("No fine-tuned checkpoints found yet.")
    print("Run the auto-discovery cell above after fine-tuning jobs complete.")